# Heatwave Exposure to Crops
Let's look around the same region - except this time we're going to keep the notebook generalized. You can always load in your own geojson to crop, but for simplicity and quick global reproducability - I am going to hardcode a bounding box in the script.

We can leverage higher resolution climate projection data from the GEE catalog. In this exercise we will be working with the [NASA_GDDP_CMIP6 dataset](https://developers.google.com/earth-engine/datasets/catalog/NASA_GDDP-CMIP6). Take a few moments to understand the various fields in this dataset. Below we will show how to use them. 


In [ ]:
import xarray as xr
import math
import geemap
import ee
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import contextily as ctx
def meters_to_decimal_degrees(meters, latitude):
    """
    Convert meters to decimal degrees based on latitude.
    
    Parameters:
    - meters (float): Distance in meters
    - latitude (float): Latitude in decimal degrees
    
    Returns:
    - float: Equivalent decimal degrees
    """
    meters_per_degree = 111320 * math.cos(math.radians(latitude))  # Adjust for latitude
    return meters / meters_per_degree

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize()




In [ ]:
# some region over Catalonia
gee_bbox = ee.Geometry.Rectangle([0.15, 40.5, 3.5, 42.9])  
variable = "tas" # starting with tas which is Daily near-surface air temperature
model = "CMCC-ESM2" # specify climate model 
map_center=[41.4, 2.1]

In [ ]:
 

# Load NASA GDDP-CMIP6 dataset for air temperature (tas)
climate_dataset = ee.ImageCollection("NASA/GDDP-CMIP6").select(variable)

# Define time periods for comparison
historical_period = climate_dataset.filter(ee.Filter.date("1975-01-01", "2000-01-01")).filter(ee.Filter.eq("model", model)).filter(ee.Filter.eq("scenario", "historical")).mean().clip(gee_bbox)
ssp585_period = climate_dataset.filter(ee.Filter.date("2075-01-01", "2100-01-01")).filter(ee.Filter.eq("model", model)).filter(ee.Filter.eq("scenario", "ssp585")).mean().clip(gee_bbox)


# Load MODIS Land Cover dataset for agriculture (Cropland)
modis = ee.ImageCollection("MODIS/061/MCD12Q1").select("LC_Type1")
latest_year = modis.sort("system:time_start", False).first()
cropland = latest_year.eq(12).clip(gee_bbox)

# Visualization parameters
temp_vis = {
    "min": 280,  # Adjust based on the dataset (temperature in Kelvin)
    "max": 300,
    "palette": ["blue", "cyan", "yellow", "orange", "red"],  # Temperature color scale
}

cropland_vis = {
    "min": 0,
    "max": 1,
    "palette": ["black", "green"],  # green for cropland
}

# Create map and add layer slider
m = geemap.Map(center=map_center, zoom=7)

# Left: Historical temperature + cropland
left_temp = geemap.ee_tile_layer(historical_period, temp_vis, "Historical (1975-2000)")
left_cropland = geemap.ee_tile_layer(cropland.selfMask(), cropland_vis, "Cropland")

# Right: SSP585 temperature + cropland
right_temp = geemap.ee_tile_layer(ssp585_period, temp_vis, "SSP585 (2075-2100)")
right_cropland = geemap.ee_tile_layer(cropland.selfMask(), cropland_vis, "Cropland")

# Create split map
m.split_map(left_temp, right_temp)

# Manually add cropland layer to both panels
m.addLayer(cropland.selfMask(), cropland_vis, "Cropland (Both Panels)")

m.add_colorbar(temp_vis, label="Avg. Air Temperature (K)", layer_name="Historical (1975-2000)")



# Display the map
m


## Heatwave impacts to vegetation
Review the slides from class and remember the optimal temperature ranges depending on crop types. So far we have only been examining binary crop layers. Now we will look at more detailed crop type maps available - in this case over Europe with [EUCROPMAP](https://developers.google.com/earth-engine/datasets/catalog/JRC_D5_EUCROPMAP_V1).

In [ ]:
# Load the latest JRC EUCROPMAP dataset
eucropmap = ee.ImageCollection("JRC/D5/EUCROPMAP/V1").filterDate("2018-01-01", "2018-12-31").first()

# Select the crop classification band
crop_classification = eucropmap.select("landcover").clip(gee_bbox)


In [ ]:
crop_reclass_map = ee.Dictionary({
    100:  0,  # Artificial    
    211:  1,  # Common wheat → Group 1 (cool-season crops)
    212:  1,  # Durum wheat → Group 1
    213:  1,  # Barley → Group 2 (moderate temperature crops)
    214:  1,  # Rye → Group 2
    215:  1,  # Oats → Group 2
    216:  3,  # Maize → Group 3 (warm-season crops)
    217:  2,  # Rice → Group 3
    218:  2,  # Triticale → Group 2
    219:  2,  # other cereals → Group 2
    221:  1,  # Potatoes → Group 3
    222:  4,  # Sugar beet → Group 4 (heat-tolerant crops)
    223:  4,  # Other root crops
    230: 0,  # other non permanent industrial crops
    231: 4,  # Sunflower → Group 4
    232: 1,  # Rapeseed → Group 4
    233: 2,  # Soybean → Group 4
    240: 5,  # Dry pulses
    250: 5,  # Fodder crops (cereals and Legumes)
    290: 5,  # Bare arable Land
    300: 0,  # Vwoodlands and shrubland
    500: 3,  # grasslands
    600: 0,  # bare land/lichens
    700: 0,  # water
    800: 0,  # wetlands
})

In [ ]:
import geemap
import ee

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Load the latest JRC EUCROPMAP dataset (2018)
eucropmap = ee.ImageCollection("JRC/D5/EUCROPMAP/V1").filterDate("2018-01-01", "2018-12-31").first()

# Select the crop classification band
crop_classification = eucropmap.select("classification").clip(gee_bbox)

# Official Color Palette from Google Earth Engine Dataset Page
crop_palette = [
    "#ff130f", "#a57000", "#896054", "#e2007c", "#aa007c",  # 100-215 (Artificial, Wheat, Barley, Rye, Oats)
    "#a05989", "#ffd300", "#00a8e2", "#d69ebc", "#d69ebc",  # 216-219 (Maize, Rice, Triticale, Other cereals)
    "#dda50a", "#a800e2", "#00af49", "#00af49", "#ffff00",  # 221-232 (Potatoes, Sugar beet, Other root crops, Industrial crops, Sunflower)
    "#d1ff00", "#267000", "#f2a377", "#e8bfff", "#696969",  # 233-290 (Rapeseed, Soya, Dry pulses, Fodder crops, Bare arable land)
    "#93cc93", "#e8ffbf", "#a89e7f", "#0793de", "#7cafaf"   # 300-800 (Woodland, Grasslands, Bare land, Water, Wetlands)
]

# Define Visualization Parameters
crop_vis = {
    "min": 100,
    "max": 800,  # The dataset contains values from 100-800
    "palette": crop_palette,
}


# Load MODIS Land Cover dataset for agriculture (Cropland)
modis = ee.ImageCollection("MODIS/061/MCD12Q1").select("LC_Type1")
latest_year = modis.sort("system:time_start", False).first()
modis_cropland = latest_year.eq(12).clip(gee_bbox)

# Visualization parameters

modis_vis = {
    "min": 0,
    "max": 1,
    "palette": ["white", "black"],  # green for cropland
}



# 🔹 Create Map and Add the Original Crop Classification Layer
m = geemap.Map(center=map_center, zoom=7)


m.addLayer(crop_classification, crop_vis, "Original EUCROPMAP (100-800)")

# Corrected Legend for EUCROPMAP
legend_dict = {
    "Artificial": "#ff130f",
    "Common wheat": "#a57000",
    "Durum wheat": "#896054",
    "Barley": "#e2007c",
    "Rye": "#aa007c",
    "Oats": "#a05989",
    "Maize": "#ffd300",
    "Rice": "#00a8e2",
    "Triticale": "#d69ebc",
    "Other cereals": "#d69ebc",
    "Potatoes": "#dda50a",
    "Sugar beet": "#a800e2",
    "Other root crops": "#00af49",
    "Other non-permanent industrial crops": "#00af49",
    "Sunflower": "#ffff00",
    "Rapeseed and turnip rapeseed": "#d1ff00",
    "Soya": "#267000",
    "Dry pulses": "#f2a377",
    "Fodder crops (cereals and leguminous)": "#e8bfff",
    "Bare arable land": "#696969",
    "Woodland and Shrubland (incl. permanent crops)": "#93cc93",
    "Grasslands": "#e8ffbf",
    "Bare land/lichens moss": "#a89e7f",
    "Water": "#0793de",
    "Wetlands": "#7cafaf",
}
m.addLayer(modis_cropland.selfMask(), modis_vis, "MODIS cropland")

# 🔹 Add Legend to Map
m.add_legend(title="EUCROPMAP Crop Types", legend_dict=legend_dict)

# Display the map
m


Examining this map it looks like the Barcelona area is filled with mostly Type 1 crops, which have an optimal temperature range of 15-20 degrees C and operative temperature range of 5-30 degrees C. These values are passed on average 5 day running means.

Let's pull out some historical and projected temperature data and calculate the difference in number of consectutive optimal days for this region and these crops.



### Convert the temperature data to xarray 

In [ ]:
# Find the native resolution of the dataset (meters)
data_resolution = climate_dataset.first().projection().nominalScale().getInfo()
print(f"Data Resolution: {data_resolution} meters")

# convert meters to a rough decimal degree

data_resolution_degrees = meters_to_decimal_degrees(data_resolution,map_center[0])
print(f"Data Resolution: {data_resolution_degrees} degrees")


In [ ]:
historical_period_time = climate_dataset.filter(ee.Filter.date("1975-01-01", "2000-01-01")).filter(ee.Filter.eq("model", model)).filter(ee.Filter.eq("scenario", "historical"))
ssp585_period_time = climate_dataset.filter(ee.Filter.date("2075-01-01", "2100-01-01")).filter(ee.Filter.eq("model", model)).filter(ee.Filter.eq("scenario", "ssp585"))


historical_ds = geemap.ee_to_xarray(
    dataset = historical_period_time,
    geometry=gee_bbox,
    scale=data_resolution_degrees,  # Preserve the original dataset resolution
    crs="EPSG:4326"  # Ensure the correct CRS
)

ssp_ds = geemap.ee_to_xarray(
    dataset = ssp585_period_time,
    geometry=gee_bbox,
    scale=data_resolution_degrees,  # Preserve the original dataset resolution
    crs="EPSG:4326"  # Ensure the correct CRS
)

In [ ]:
historical_ds

In [ ]:
# Apply 5-day rolling mean
historical_rolling = historical_ds.rolling(time=5, center=True).mean()
ssp_rolling = ssp_ds.rolling(time=5, center=True).mean()



### Apply operative temperature range 
Make sure to account for the climate projection data coming in Kelvin

In [ ]:
# For C3 or Type 1 Plants in this region
optimal_min = 15 
optimal_max = 20

In [ ]:
# Count occurrences where temperature is within the range at each grid cell
historical_count = ((historical_rolling['tas'] >= optimal_min+273) & (historical_rolling['tas'] <= optimal_max+273)).sum(dim="time")
ssp_count = ((ssp_rolling['tas'] >= optimal_min+273) & (ssp_rolling['tas'] <= optimal_max+273)).sum(dim="time")
historical_count

In [ ]:
count_difference = ssp_count - historical_count
count_difference

Now let's use the MODIS crop binary mask and downscale these optimal day counts to the same grid and plot.

In [ ]:
# Find the native resolution of the dataset (meters)
modis_data_resolution = modis.first().projection().nominalScale().getInfo()
print(f"Data Resolution: {modis_data_resolution} meters")

# convert meters to a rough decimal degree

modis_data_resolution_degrees = meters_to_decimal_degrees(modis_data_resolution,map_center[0])
print(f"Data Resolution: {modis_data_resolution_degrees} degrees")

In [ ]:
crop_mask_ds = geemap.ee_to_xarray(
    dataset = modis_cropland,
    geometry=gee_bbox,
    scale=modis_data_resolution_degrees,  # Preserve the original dataset resolution
    crs="EPSG:4326"  # Ensure the correct CRS
)

In [ ]:
crop_mask_ds

In [ ]:
downscaled_count_difference = count_difference.interp(
    lat=crop_mask_ds.lat, 
    lon=crop_mask_ds.lon, 
    method="nearest"  # Other options: "nearest", "cubic"
)
downscaled_count_difference

Apply crop mask

In [ ]:
downscaled_count_difference_masked = downscaled_count_difference * crop_mask_ds['LC_Type1'][0]
downscaled_count_difference_masked = downscaled_count_difference_masked.transpose("lat", "lon")
downscaled_count_difference_masked.plot()

In [ ]:
# Create the figure
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})


# Set the extent (bounding box of netcdf data plus some buffer)
buffer = 0.25 
lon_min, lon_max = downscaled_count_difference_masked.lon.min().item(), downscaled_count_difference_masked.lon.max().item()
lat_min, lat_max = downscaled_count_difference_masked.lat.min().item(), downscaled_count_difference_masked.lat.max().item()


ax.set_extent([lon_min-buffer, lon_max+buffer, lat_min-buffer, lat_max+buffer], crs=ccrs.PlateCarree())

# Define colormap range
vmin, vmax = -1000, 1000

# Plot the average difference for the selected month
pcm = downscaled_count_difference_masked.plot(ax=ax, cmap="RdBu_r", alpha=0.7, vmin=vmin, vmax=vmax, add_colorbar=False)

# Add a title
#ax.set_title(f'SPEI{spei_dur} Month {spei_month}: {gcm.upper()} {scenario.upper()} ({start_year}-{end_year})', fontsize=16)

# Add coastlines and basemap
ax.add_feature(cfeature.COASTLINE)
ctx.add_basemap(ax, crs=historical_ds.crs,
                source=ctx.providers.OpenStreetMap.Mapnik, attribution_size=6, alpha=1)

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 12}
gl.ylabel_style = {"size": 12}

# Add colorbar
cbar = fig.colorbar(pcm, ax=ax, location="bottom", shrink=0.7, label="# of Days")
cbar.ax.tick_params(labelsize=12)
ax.set_title("Change in Optimal Temperature Range Days for Crop Areas", fontsize=16, fontweight="bold")

plt.savefig('heatwave-crop-changes.png')

plt.show()
